In [ ]:
import ast
import string

import pandas as pd

DATASET_PATH = "../../data/out/distillation/mmlu_distilled_deepseek_v4_flash_extended.parquet"

df = pd.read_parquet(DATASET_PATH)
print(f"Path: {DATASET_PATH}")
print(f"Shape: {df.shape}")
df.head(2)

: 

## Accuracy

In [ ]:
n = len(df)
n_correct = int(df["distill_ans_correct"].sum())
print(f"Accuracy: {n_correct / n:.4f} ({n_correct}/{n})")

## Missing distilled reasoning traces

In [ ]:
missing_nan = int(df["distill_reasoning"].isna().sum())
empty_mask = df["distill_reasoning"].fillna("").str.strip() == ""
missing_or_empty = int(empty_mask.sum())
print(f"NaN distill_reasoning:            {missing_nan}")
print(f"Missing or empty distill_reasoning: {missing_or_empty}")

## Reasoning trace length distribution (chars)

In [ ]:
existing_reasoning = df.loc[~empty_mask, "distill_reasoning"]
existing_reasoning.str.len().describe()

## Invalid answers

An answer is valid iff, after stripping/lowercasing, it is one of the option letters `a`, `b`, ... up to the row's number of options.

In [ ]:
LETTERS = list(string.ascii_lowercase)


def n_options(opts):
    if isinstance(opts, str):
        try:
            return len(ast.literal_eval(opts))
        except Exception:
            return 0
    try:
        return len(opts)
    except TypeError:
        return 0


def is_valid_answer(ans, n_opts):
    if not isinstance(ans, str):
        return False
    return ans.strip().lower() in set(LETTERS[:n_opts])


n_opts_series = df["options"].apply(n_options)
valid_mask = pd.Series(
    [is_valid_answer(a, k) for a, k in zip(df["distill_answer"], n_opts_series)],
    index=df.index,
)
n_invalid = int((~valid_mask).sum())
print(f"Invalid distill_answer count: {n_invalid} ({n_invalid / n:.2%})")

### All invalid answers

In [ ]:
invalid_answers = df.loc[~valid_mask, "distill_answer"]
for idx, ans in invalid_answers.items():
    print(f"--- row {idx} ---")
    print(repr(ans))

### 10 shortest reasoning traces

In [ ]:
shortest = existing_reasoning.str.len().nsmallest(10)
for idx, length in shortest.items():
    print(f"--- row {idx} (len={length}) ---")
    print(df.loc[idx, "distill_reasoning"])
    print()

## Reset distill columns for invalid answers

Sets `distill_reasoning` and `distill_answer` to `""` and `distill_ans_correct` to `False` for rows where the answer is invalid, then writes back to `DATASET_PATH`.

In [ ]:
# invalid_idx = df.index[~valid_mask]
# df.loc[invalid_idx, "distill_reasoning"] = ""
# df.loc[invalid_idx, "distill_answer"] = ""
# df.loc[invalid_idx, "distill_ans_correct"] = False

# df.to_parquet(DATASET_PATH, index=False)
# print(f"Reset {len(invalid_idx)} rows and wrote {DATASET_PATH}")